In [ ]:
######################################################
# ! 11. Agents - Why a fixed RAG pipeline isn't enough
######################################################

In [25]:
# Создаем в этой папке два файла: rag_helper и ingest

!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [26]:
# Код для работы с API OpenAI, а также для 
# загрузки переменных окружения из файла .env.

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [27]:
# - load_faq_data() - Обращаемся к апи и получаем список курс,
# затем для каждого курса  загружает отдельный JSON‑файл 
# вопросами и ответами, объединяет  все эти данные в один
# большой список словарей (документов) и возвращает его.

# - build_index принимает полученный список документов и 
# создаёт инвертированный индекс с помощью библиотеки minsearch. 
# Этот индекс позволяет искать по текстовым полям (question, section, 
# answer) и фильтровать по ключевому полю (course). После вызова
# индекс готов к выполнению поисковых запросов (например, 
# index.search("как записаться на курс", filter_dict={"course": "mlops"})).

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
###############################################################
# ! 13. Function Calling - дать ЛЛМ возможность самому
# ! принимать решения, а не следовать жесткому сценарию
###############################################################
# ? Проблема обычного RAG заключается в том, что пользователь 
# ? задаёт вопрос. Система ищет ответ в базе знаний. Результат
# ? поиска представляются в prompt. LLM дает ответ
# ! Пользователь → Поиск → LLM → Ответ
###############################################################


In [28]:
# Этот код создаёт ассистента на основе RAG (Retrieval-Augmented Generation) 
# — системы, которая ищет релевантные фрагменты из загруженного FAQ и 
# передаёт их в большую языковую модель (GPT), чтобы она дала ответ 
# на вопрос пользователя, опираясь только на факты из базы знаний.

from rag_helper import RAGBase

instructions = """
  You're a course teaching assistant.
  Answer the QUESTION based on the CONTEXT from the FAQ database.
  Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
  index = index,
  llm_client = openai_client,
  instructions = instructions,
)

In [33]:
answer = assistant.rag('How do I run Olama locally?')
# print(answer) => I don’t see any FAQ entry for running **Olama** locally.

answer = assistant.rag('How do I run Ollama locally?')
# print(answer) => To run Ollama locally: ...

In [ ]:
# При вызове .rag() произойдёт:

# 1. Поиск по индексу с фильтром по курсу llm-zoomcamp (по умолчанию) 
# и бустом полей question (выше) и section (ниже).
# 2. Формирование контекста — из найденных документов извлекаются секции, 
# вопросы и ответы.
# 3. Построение промпта — подстановка вопроса и контекста в шаблон.
# 4. Отправка в OpenAI — сообщение с инструкциями (разработчик) 
# и пользовательским промптом.
# 5. Получение ответа — модель возвращает текст, который возвращается 
# как результат rag().

In [ ]:
# В этом примере используется прямой вызов к OpenAI API (без RAG и без доп.инструкций)
messages = [
  { 'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
  model = 'gpt-5.4-mini',
  input = messages,
)

response.output_text

# 'Yes — if the course is still open, you can usually join it.\n\n
# If you just discovered it, the next step is to check:\n- whether 
# enrollment is still available,\n- whether there are any prerequisites,
# \n- and whether there’s a deadline or waiting list.\n\nIf you want, 
# I can help you figure out the best way to ask the instructor or course
# organizer.

In [ ]:
# Функция search(query) выполняет поиск по заранее построенному индексу 
# (переменная index, которая была создана ранее через build_index(documents))
# Она возвращает список релевантных документов
# (вопросов и ответов) для заданного пользовательского запроса.

def search(query):
  boost_dict = {'question': 3.0, 'section': 0.5}
  filter_dict = {'course': 'llm-zoomcamp'}

  return index.search(
    # Принимает вопрос пользователя
    query,
    # Возвращает не более 5 найденных
    num_results = 5,
    # Задает вес
    boost_dict = boost_dict,
    # Ищет только в курсе 'llm-zoomcamp'
    filter_dict = filter_dict
  )

In [ ]:

# JSON-схема для больших языковых моделей

search_tool = {
  "type": "function",
  "name": "search",
  "description": "Search the FAQ database for entries matching the given query.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Search query text to look up in the course FAQ."
      }
    },
    "required": ["query"],
    "additionalProperties": False
  }
}

In [ ]:
# Велючаем схему в тулзы

response = openai_client.responses.create(
  model='gpt-5.4-mini',
  input = messages,
  tools=[search_tool]
)

len(response.output) # 1
response.output[0] 

# ResponseFunctionToolCall(arguments='{"query":
# "join course discovered late can I join enroll late join course"}',
# call_id='call_UpV80e7ba8LoEnDqsjqAfJeC', 
# name='search', 
# type='function_call', 
# id='fc_0f4fa185156188ae006a156dad12e481938c171d34718b305f', 
# namespace=None, 
# status='completed'
# )

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join"}', call_id='call_R0xq66tgpgSkahLxIV79fxc8', name='search', type='function_call', id='fc_0ce10d168fcfdcb7006aa1ca17fe5487d18b617dec02a98d74', async_=None, caller=None, namespace=None, status='completed')

In [ ]:
# Обратите внимание и на аргументы. Модель не приняла наш вопрос 
# дословно. Она посчитала, что исходный вопрос не является
# оптимальным поисковым запросом. Поэтому она переформулировала
# наш вопрос о зачислении в поисковые ключевые слова, например
# «записаться на курс с поздним поступлением».

In [ ]:
# Выполнение функции и отправка результата обратно.
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent = 2)

# {'query': 'join course discovered late can I join enroll late join course'}

In [ ]:
# Теперь мы отправляем этот результат обратно в модель. Сначала мы 
# добавляем вывод модели в историю диалога — модели необходимо 
# увидеть собственный вызов функции. Затем мы добавляем результат 
# работы инструмента.
messages.extend(response.output)

messages.append({
  "type": "function_call_output",
  "call_id": call.call_id,
  "output": result_json,
})

# Этот call_id инструмент связывает результат выполнения с 
# конкретным вызовом функции, запрошенным моделью. Если 
# модель выполняет несколько вызовов функций за один раз, 
# каждому из них присваивается свой собственный идентификатор 
# call_id.

In [20]:
messages.append(call)

In [ ]:
# ! Подсчет траты токенов

In [ ]:
# Чтобы узнать сколько токенов используется на 1 запрос необходимо узнать: 
# - Сколько входных и выходных токенов
usage = response.usage                          # 653
usage.input_tokens, usage.output_tokens         # 33

In [ ]:
# Теперь использую информацию из токенов мы можем рассчитать эту стоимость

def calculate_gpt54mini_price(input_tokens, output_tokens):
  INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
  OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

  input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
  output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

  total_cost = input_cost + output_cost

  return {
    "input_cost": input_cost,
    "output_cost": output_cost,
    "total_cost": total_cost
  }

result = calculate_gpt54mini_price(652, 33)
print("Total Cost: $", round(result["total_cost"], 8))

# Total Cost: $ 0.0001176

Total Cost: $ 0.0001176


In [ ]:
##########################################################
# ! 14. The Agentic Loop - алгоритм работы ИИ, при котором
# ! модель выполняет задачу в несколько цикличных шагов
# ! без участия человека.

# ! Анализ задачи ➔ Действие (например, поиск в сети) ➔ 
# ! Проверка результата ➔ Коррекция и повтор.

# ! Цикл крутится до тех пор, пока цель не будет 
# ! полностью достигнута.
##########################################################
# Агент состоит из 3 частей:
# - Инструкции, роль и желаемое поведение. 
# - Инструменты - функция (search), которые вызывает агент 
# для выполнения задач
# - Память, история сообщений. Мы добавляем каждый запрос, 
# каждый результат работы модели и каждый результат работы 
# инструмента. Агент читает это, чтобы знать, что он уже
# пытался сделать.

In [37]:
# 1. Инструкция

instructions = """
  You're a course teaching assistant.
  You're given a question from a course student and your task is to answer it.

  If you want to look up information, use the search function. 
  Use as many keywords from the user question as possible when making first requests.

  Make multiple searches.

  Try to expand your search by using new keywords
  based on the results you get from the search.

  At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can i join it?'

messages = [
  {'role': 'developer', 'content': instructions},
  {'role': 'user', 'content': question}
]

In [38]:
search_tool = {
  "type": "function",
  "name": "search",
  "description": "Search the FAQ database for entries matching the given query.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Search query text to look up in the course FAQ."
      }
    },
    "required": ["query"],
    "additionalProperties": False
  }
}

In [ ]:
response = openai_client.responses.create(
  model = 'gpt-5.4-mini',
  # 2. Память (помнит о пред.действиях)
  input = messages,
  # 3. Инструменты
  tools = [search_tool]
)

''

In [45]:
# Преобразует строку JSON - в словарь пайтон
# выводит данные с помощью output
def make_call(call):
  args = json.loads(call.arguments)

  if call.name == "search":
    result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
      "type": "function_call_output",
      "call_id": call.call_id,
      "output": result_json,
    }

In [ ]:
# Сохраняем все в истории
messages.extend(response.output)


# В ответе мы можем получить много элементов, нам необходимо перебрать их

for item in response.output:
  # Проверяем к какому типу относится элемент => type='function_call'
  if item.type == 'function_call':
    print("Вызов функции:_", item.name, 'и имя элемента:_', item.arguments)
    call_output = make_call(item)
    messages.append(call_output)

  # Но если тип сообщения, то пропускаем пока что шаг  
  elif item.type == 'message':
    print("ASSISTANT:")
    print(item.content[0].text)


Вызов функции:_ search и имя элемента:_ {"query":"join course discovered course can I join enrollment access registration FAQ"}


In [ ]:
messages

'''
[
  ----------- ЧТО ИМЕННО ДЕЛАЕТ АГЕНТ --------------
  {
    
    'role': 'developer',
    'content': 
      "\n  You're a course teaching assistant.\n  You're given a question from a course student and your task is to answer it.\n\n  
      If you want to look up information, use the search function. \n  Use as many keywords from the user question as possible when making 
      first requests.\n\n  Make multiple searches.\n\n  Try to expand your search by using new keywords\n  
      based on the results you get from the search.\n\n  At the end, ask if there are other areas that the user wants to explore.\n"
  },
  ----------- ВОПРОС ОТ СТУДЕНТА ----------------
  {
    'role': 'user', 
    'content': 'I just discovered the course. Can i join it?'
  },
  ---------- ВЫЗОВ ДВУХ ЭЛЕМЕНТОВ, КОТОРЫЙ LLM ИСПОЛЬЗУЕТ ДЛЯ ВЫЗОВА ИНСТРУМЕНТА И РЕЗУЛЬТАТА. Это то что мы отправим обратно -----------------
  ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment access registration FAQ"}', call_id='call_3dJeoZqc2kOugW7iE0Y5DuVi', name='search', type='function_call', id='fc_02edd1232f633f2b006aa3bca1104487d1a4d384205ace6d9b', async_=None, caller=None, namespace=None, status='completed'),
  ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment access registration FAQ"}', call_id='call_3dJeoZqc2kOugW7iE0Y5DuVi', name='search', type='function_call', id='fc_02edd1232f633f2b006aa3bca1104487d1a4d384205ace6d9b', async_=None, caller=None, namespace=None, status='completed'),
  {
    'type': 'function_call_output',
    'call_id': 'call_3dJeoZqc2kOugW7iE0Y5DuVi',
    'output': '[\n  
      {\n    "id": "74eb249bbf",\n   
        "course": "llm-zoomcamp",\n    
        "section": "General Course-Related Questions",\n    
        "question": "I just discovered the course. Can I still join?",\n    
        "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  
      },\n  
      {\n    
        "id": "5cc511f85b",\n    
        "course": "llm-zoomcamp",\n    
        "section": "General Course-Related Questions",\n    
        "question": "Does the course certificate show the number of course hours?",\n    
        "answer": "No. The certificate does not state a total number of hours."\n  
      },\n  
      {\n    
        "id": "e2d595f23c",\n    
        "course": "llm-zoomcamp",\n    
        "section": "General Course-Related Questions",\n    
        "question": "Why is the number of documents in the FAQ dataset different from the video, and why do my RAG results differ?",\n    
        "answer": 
          "The course loads documents from the live FAQ dataset, which changes over time as\\nquestions are added, 
          updated, or deleted. If your notebook downloads the latest\\ndata, its document count and RAG index can
          differ from the snapshot used when the\\nvideos were recorded. Different retrieved context can then 
          produce a different\\nfinal answer.\\n\\nThis does not necessarily mean your implementation is wrong. 
          To reproduce a\\nvideo exactly, use the same dataset snapshot or Git commit; otherwise, expect\\nresults
          from the current dataset to differ."\n  
      },\n  
      {\n   
        "id": "977bf7786c",\n    
        "course": "llm-zoomcamp",\n    
        "section": "General Course-Related Questions",\n    
        "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    
        "answer": 
          "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework 
          (while the form is open) without registering. It is not checked against any registered list. 
          Registration is just to gauge interest before the start date."\n  
      },\n  
      {\n    
        "id": "69d122f12e",\n    
        "course": "llm-zoomcamp",\n    
        "section": "General Course-Related Questions",\n    
        "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n   
        "answer": 
          "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nTo get the 
          certificate, you need to finish a capstone project and complete the\\nrequired peer reviews. Homework 
          is not required. You can work through the\\nmaterial and prepare your project in self-paced mode, but 
          project submission and\\npeer review must happen while a live cohort is accepting them."\n  
      }\n]'}]
'''

[{'role': 'developer',
  'content': "\n  You're a course teaching assistant.\n  You're given a question from a course student and your task is to answer it.\n\n  If you want to look up information, use the search function. \n  Use as many keywords from the user question as possible when making first requests.\n\n  Make multiple searches.\n\n  Try to expand your search by using new keywords\n  based on the results you get from the search.\n\n  At the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can i join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment access registration FAQ"}', call_id='call_3dJeoZqc2kOugW7iE0Y5DuVi', name='search', type='function_call', id='fc_02edd1232f633f2b006aa3bca1104487d1a4d384205ace6d9b', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join course discovered

In [52]:
# Теперь сохраняем все в один файл
it = 1    # Номер итерации

while True:
    print(f"iteration #{it}... - сколько запросов мы отправили")
    has_function_calls = False

    response = openai_client.responses.create(
      model="gpt-5.4-mini",
      input=messages,
      tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    # Если вызова функции нет, мы прерываем цикл
    it = it + 1
    if has_function_calls == False:
        break

iteration #1... - сколько запросов мы отправили
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open. You can also start learning right away; the videos and materials are available.

If you’d like, I can also point you to the best place to start in the course materials. Is there anything else you want to explore?


In [9]:
# И помещаем все в функцию и возвращаем последний ответ item.content[0].text
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model = model,
            input = messages,
            tools = [search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [10]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


NameError: name 'openai_client' is not defined

In [55]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join enroll late FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, just make sure you submit your project while submissions are still being accepted.

Would you like me to help with anything else about the course?


'Yes — you can still join the course.\n\nIf you want to receive a certificate, just make sure you submit your project while submissions are still being accepted.\n\nWould you like me to help with anything else about the course?'

In [ ]:
agent_loop(instructions, "what's queen gambit?")

In [8]:
# Не отвечай на вопросы если они не по теме
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

NameError: name 'agent_loop' is not defined

In [ ]:
##########################################################
# ! 15. ToyAIKit - это фреймворк для создания Ai-agents, 
# ! если раньше мы писали его вручную для каждого запроса, то 
# ! теперь мы можем просто исп библиотеку
##########################################################

In [3]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [4]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [5]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [6]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [15]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
# Интерфейс чата и runner

# The chat_interface handles display in the notebook.
chat_interface = IPythonChatInterface()

# The callback renders model messages and tool calls as they happen. 
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
  tools = agent_tools,
  developer_prompt = instructions,
  chat_interface = chat_interface,
  llm_client = OpenAIClient(model="gpt-5.4-mini")
)

In [17]:
result = runner.loop(
  prompt = "How do I run Olama locally?",
  callback = callback,
)

APITimeoutError: Request timed out.

In [18]:
result.cost

NameError: name 'result' is not defined

In [ ]:
result.all_messages